
# 38 因子流水线：从数据到可复现报告

> **本章要解决的问题**：同一个因子、同一份代码，两个人跑出来的 IC 为什么不一样？某个数字出错时，怎么证明“它就是那次运行的结果”？任务失败了，是立刻重跑、等一会儿再重试，还是直接放弃？

## 本章导读

前 37 章都在回答“因子有没有用”：01 讲因子与标签，03 讲单因子报告，11 讲 WT 单因子流水线，17 讲生命周期，29 讲多重检验。这一章换一个视角，不问“因子好不好”，只问“**这条产线稳不稳**”。

一个可复现的因子流水线，至少要回答五个问题：

1. 输入是什么：数据契约、字段、时间语义、PIT 快照；
2. 中间怎么算：因子卡、计算 DAG、节点指纹；
3. 任务怎么跑：队列、幂等键、状态机、失败重试；
4. 结果怎么存：版本、报告 manifest、可追溯字段；
5. 出问题怎么查：状态留痕、死信、版本 diff。

### 学习目标

- 给定一份行情或财务数据和字段说明，能写出数据契约（字段、主键、时间语义、缺失规则），并把它落成可执行的校验。
- 给定一次因子计算任务，能把输入冻结成内容 hash，并用 hash 判断两次运行是不是同一份数据。
- 给定若干计算步骤及其依赖，能构造简易 DAG，按拓扑序执行，并在重跑时跳过指纹未变的节点。
- 给定一个会间歇失败的节点，能设计重试策略（最大次数、指数退避、死信），并把状态迁移写进作业表。
- 给定一次成功运行，能生成报告 manifest（代码版本、快照 hash、参数 hash、指标、作业号），并用它复现或审计。
- 给定一份报告，能说出它由哪个代码版本、哪份数据、哪套参数、哪些作业产生。

### 先修

第12章的因子流水线；第19章的 PIT 与“当时可见的数据”；第20章的多重检验；Python 标准库 `sqlite3`、`json`、`hashlib`；会 numpy 的基本操作。

## 在量化研究中的位置

- **谁用**：因子研究员、数据工程、平台维护、风控与合规、接手代码的同事。
- **输入**：行情/财务/另类数据、字段说明、因子表达式或代码、参数、交易日历。
- **输出**：可复现的因子面板、运行日志、报告 manifest、失败记录与死信。
- **在流程的哪一步**：位于“数据入库”之后、“单因子报告/组合”之前；它把一次性脚本变成可重复调用的产线。
- **失败模式**：用文件名或日期当版本、快照被悄悄覆盖、重跑不幂等、失败后无限重试、报告里的数字找不到来源。
- **最小工作流**：写数据契约 → 冻结快照 hash → 建 DAG → 入队执行（幂等）→ 失败重试/死信 → 出报告 manifest → 归档。

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="38 因子流水线：从数据到可复现报告" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>38 因子流水线：从数据到可复现报告｜知识图</title>
<desc>本章主线：数据契约 → 计算 DAG → 幂等队列 → 报告 manifest，整条链可重跑、可审计。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a38" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao38" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag38" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">38</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">因子流水线：从数据到可复现报告</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：数据契约 → 计算 DAG → 幂等队列 → 报告 manifest，整条链可重跑、可审计。</text>
<rect x="65" y="168" width="125" height="66" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="127.5" y="193.8" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">原始快照（PIT）</text>
<text x="127.5" y="212" font-size="8.6" fill="#6E7B8B" text-anchor="middle">hash 冻结</text>
<rect x="245" y="168" width="125" height="66" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="307.5" y="193.8" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">清洗 / 对齐</text>
<text x="307.5" y="212" font-size="8.6" fill="#6E7B8B" text-anchor="middle">频率、日历、缺失</text>
<rect x="430" y="100" width="130" height="66" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="495" y="125.8" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">因子 A 计算</text>
<text x="495" y="144" font-size="8.6" fill="#6E7B8B" text-anchor="middle">DAG 节点</text>
<rect x="430" y="236" width="130" height="66" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="495" y="261.8" font-size="12" fill="#1B3A5C" text-anchor="middle" font-weight="bold">因子 B 计算</text>
<text x="495" y="280" font-size="8.6" fill="#6E7B8B" text-anchor="middle">并行分支</text>
<rect x="640" y="168" width="122" height="66" fill="#1B3A5C" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="701" y="193.9" font-size="12.5" fill="#FFFFFF" text-anchor="middle" font-weight="bold">合并报告</text>
<text x="701" y="212.3" font-size="8.6" fill="#FFFFFF" text-anchor="middle">版本/参数/快照</text>
<path d="M 190 201 L 245 201" fill="none" stroke="#E07B39" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ao38)"/>
<path d="M 370 201 L 430 133" fill="none" stroke="#E07B39" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ao38)"/>
<path d="M 370 201 L 430 269" fill="none" stroke="#1B3A5C" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a38)"/>
<path d="M 560 133 L 640 201" fill="none" stroke="#E07B39" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ao38)"/>
<path d="M 560 269 L 640 201" fill="none" stroke="#1B3A5C" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a38)"/>
<rect x="195" y="185" width="46" height="14" fill="#FFFFFF"/>
<text x="218" y="194" font-size="10" fill="#6E7B8B" text-anchor="middle">数据契约</text>
<rect x="374.4" y="151" width="51.3" height="14" fill="#FFFFFF"/>
<text x="400" y="160" font-size="10" fill="#6E7B8B" text-anchor="middle">DAG 调度</text>
<rect x="377" y="241" width="46" height="14" fill="#FFFFFF"/>
<text x="400" y="250" font-size="10" fill="#6E7B8B" text-anchor="middle">并行分支</text>
<rect x="587" y="149" width="26" height="14" fill="#FFFFFF"/>
<text x="600" y="158" font-size="10" fill="#6E7B8B" text-anchor="middle">合并</text>
<rect x="587" y="235" width="26" height="14" fill="#FFFFFF"/>
<text x="600" y="244" font-size="10" fill="#6E7B8B" text-anchor="middle">合并</text>
<line x1="650" y1="96" x2="668" y2="96" stroke="#E07B39" stroke-width="2.4" stroke-linecap="round"/>
<text x="673" y="100" font-size="9" fill="#E07B39">关键路径</text>
<rect x="41" y="322" width="718" height="62" fill="#E6EFF7" rx="14"/>
<text x="58" y="344" font-size="9.4" fill="#1B3A5C">幂等：同一快照 + 同一参数 → 同一结果；失败 → 指数退避重试，断点可续跑</text>
<text x="58" y="364" font-size="9.4" fill="#6E7B8B">报告 manifest.json：版本 / 参数 / 快照 hash / 指标，整条链可重跑、可审计</text>
</svg>

> **图**：本章主线：数据契约 → 计算 DAG → 幂等队列 → 报告 manifest，整条链可重跑、可审计。


In [1]:

import hashlib
import json
import sqlite3

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("sqlite3", sqlite3.sqlite_version, "| numpy", np.__version__)
print("matplotlib 中文字体已设置")


sqlite3 3.51.2 | numpy 2.4.4
matplotlib 中文字体已设置



## 1. 数据契约与 PIT 快照

**场景**：周一早上，研究员小周拿到一份“沪深 300 日行情”。他写了两行代码读入 CSV，然后直接算 20 日反转。下午同事用同一个脚本重跑，IC 差了 0.02。查了两小时才发现：这次的 CSV 多了一列复权因子，`close` 的含义从“不复权价”变成了“后复权价”。

因子流水线的第一道关，不是算因子，而是**约定数据**。这份约定叫数据契约（data contract）。

### 1.1 契约要写哪几项

| 项目 | 例子 | 不写会怎样 |
|---|---|---|
| 字段名/类型 | `trade_date: date`、`close: float` | 读进来是字符串，排序出错 |
| 主键 | `(trade_date, asset)` | 同一行被写入两次 |
| 时间语义 | `trade_date` 是交易日，`knowledge_date` 是可见日 | 用了当时还看不到的数据 |
| 单位/复权 | `close` 为后复权价 | 复权口径一变，收益断裂 |
| 缺失规则 | 停牌日不写行，也不补 0 | 0 收益污染因子 |
| 取值范围 | `close > 0`、`volume >= 0` | 负价格进账 |
| 更新频率 | 日频，T+1 入库 | 快照被悄悄覆盖 |

关键点：**契约必须能执行**。写在 README 里的字段表只能提醒人，写在代码里的断言才能拦住机器。常见做法是把契约写成一个字典，让校验函数逐条返回“通过/不通过”，失败就停在这道关，而不是把脏数据送进 DAG。

### 1.2 PIT 快照：数据也有“当时可见”

同一份 2023 年年报，在 2024 年 3 月看到的是“已披露版本”，在 2024 年 8 月看到的是“追溯调整版本”。如果回测时用后者去算 2024 年 4 月的因子，就等于偷看了未来。第19章把这件事讲成“库存修订与当时可见的数据”；在流水线里，它落成一个字段：`knowledge_date`（这条数据从哪一天开始可见）。

PIT（point-in-time）快照有两种常见做法：

1. 每次入库只追加、不覆盖，查询时取 `knowledge_date <= 回测日` 的最新一条；
2. 每次入库生成一个不可变快照，快照名就是内容 hash，历史快照只读。

两种做法可以叠加：快照冻结 + 行级 `knowledge_date`。本章代码用第二种的简化版——把数据规范化后用 hash 命名快照；第40章再把行级 PIT 做成数据库表。

### 1.3 快照 hash：判断“是不是同一份数据”

用文件名当版本是最常见的坑：`data_20240105.csv` 可能被覆盖，`最新.csv` 甚至不知道是哪天的。可靠的做法是**对内容做 hash**：

1. 把每行数据转换成规范字符串；
2. 对行排序，消除行序影响；
3. 用 `sha256` 得到摘要，取前 16 位作为快照 ID。

这样内容变一个字节，hash 就变；内容和行序都相同，hash 就一定相同。报告里记下这个 hash，就能回答“这份报告用的是哪份数据”。

### 1.4 代码：契约校验 + 快照 hash

下面这段代码做三件事：用规范字符串算快照 hash、用契约字典做四条断言、演示追加一行后 hash 必变。



In [2]:

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


raw_rows = [
    "2024-01-02,600000,10.21,1234567",
    "2024-01-03,600000,10.35,998877",
    "2024-01-02,000001,9.87,765432",
]
fields = ["trade_date", "asset", "close", "volume"]
rows = [dict(zip(fields, line.split(","))) for line in raw_rows]

snap_hash = sha256_text("\n".join(sorted(raw_rows)))
print("行数", len(rows), "| 快照 hash", snap_hash)
snap_hash2 = sha256_text("\n".join(sorted(raw_rows + ["2024-01-03,000001,9.91,443322"])))
print("追加一行后 hash", snap_hash2, "| 与旧 hash 相同:", snap_hash2 == snap_hash)

contract = {
    "fields": fields,
    "keys": ["trade_date", "asset"],
    "close_min": 0.0,
    "volume_min": 0,
}
checks = {
    "字段完整": all(set(r) == set(contract["fields"]) for r in rows),
    "主键唯一": len({(r["trade_date"], r["asset"]) for r in rows}) == len(rows),
    "收盘价>0": all(float(r["close"]) > contract["close_min"] for r in rows),
    "成交量>=0": all(int(r["volume"]) >= contract["volume_min"] for r in rows),
}
print("契约校验", checks)
print("通过", sum(checks.values()), "/", len(checks))


行数 3 | 快照 hash be88967385d09b1a
追加一行后 hash 8834fec312d377cf | 与旧 hash 相同: False
契约校验 {'字段完整': True, '主键唯一': True, '收盘价>0': True, '成交量>=0': True}
通过 4 / 4



**读数**：三行数据的快照 hash 是 `be88967385d09b1a`；追加一行后变成 `8834fec312d377cf`，说明 hash 确实跟着内容走。契约四条断言全部通过（4/4）——注意这里只演示了“能通过的契约”，真实项目里还要准备一份**故意违规**的样本，确认校验会失败，否则你不知道校验到底有没有生效。

**常见错误**：

1. 用文件修改时间或文件名当版本——文件可能没变，也可能被覆盖。
2. 直接对原始 CSV 全文 hash，而不做行排序——同一份数据换个导出顺序就变成“新快照”。
3. 契约只写文档不写断言——文档没人看，断言会拦住你。

---



## 2. 因子卡：把一个因子写成一页

**场景**：半年后有人问你，“rev20 这个因子当时怎么定义的？方向是正是负？用的哪个股票池？”如果你答不上来，这个因子就只剩一堆没人敢动的代码。

因子卡（factor card）是流水线里最便宜、收益最高的文档。它不是论文，是**交接单**。

| 字段 | 例子 | 作用 |
|---|---|---|
| 名称/ID | `rev20` / `F-0017` | 唯一标识 |
| 一句话定义 | 过去 20 日收益取负 | 防止口径漂移 |
| 方向 | -1（越小越好） | 合成时不能反 |
| 窗口/频率 | 20 日 / 日频 | 复现输入 |
| 宇宙 | csi300，按月末成分 | 决定截面 |
| 预处理 | 去极值 3σ、截面 zscore | 决定数值 |
| 标签 | 未来 20 日收益 | 决定 IC 口径 |
| 数据依赖 | close（后复权） | 绑定快照 |
| 评价指标 | IC/ICIR/换手/分组单调 | 验收标准 |
| 状态 | 草稿/验证中/观察/退役 | 生命周期 |
| 负责人/日期 | 小周 / 2024-01-05 | 找谁问 |
| 代码入口 | `pipeline.py::compute_rev20` | 在哪跑 |

因子卡有三条规矩：第一，**方向和单位必须写死**，不能靠“代码里乘了个 -1”来记忆；第二，**口径变更要升版本**，不要悄悄改；第三，**因子卡和代码同仓**，改代码的那次提交必须同时改卡片。

**常见错误**：

1. 只写公式不写宇宙/频率——换个股票池，这就不是同一个因子。
2. 方向靠口头约定——合成时一半人加、一半人减。
3. 因子卡和实现分家——卡片写 A，代码跑 B。

---

## 3. 计算 DAG：先定依赖，再谈并行

**场景**：一份因子报告要跑六步：读快照 → 清洗 → 算因子 → 对齐股票池 → 算 IC/分组 → 出报告。有人把所有逻辑塞进一个大函数，改任何一步都要从头跑；有人把“算 IC”和“算因子”并行，结果读到了还没算完的文件。

DAG（有向无环图）把这件事说清楚：**节点**是一次计算，**边**是依赖，**拓扑序**是合法的执行顺序。图里一旦有环，说明定义本身矛盾——比如“IC 依赖因子，因子又依赖 IC 的筛选结果”，这时程序只能死循环或靠人为约定，必须先把环拆掉。

### 3.1 节点指纹：增量重算的基础

每个节点算一个指纹：`hash(节点名 + 上游指纹 + 参数 hash + 代码版本)`。上游变了，指纹跟着变；参数变了，指纹也变；全都没变，结果可以直接复用。这样“重跑”就变成“只算变了的部分”，既省时间，又保证新结果和旧结果口径一致。

指纹还有一个好处：它把“我认为没变”变成“可以验证没变”。报告里记下最终节点的指纹，下次重跑先比指纹，再决定要不要重新出报告。

### 3.2 代码：拓扑排序 + 指纹复用



In [3]:

nodes = {
    "load": [],
    "clean": ["load"],
    "factor": ["clean"],
    "universe": ["clean"],
    "test": ["factor", "universe"],
    "report": ["test"],
}


def topo_order(graph):
    indeg = {n: len(deps) for n, deps in graph.items()}
    children = {n: [] for n in graph}
    for n, deps in graph.items():
        for d in deps:
            children[d].append(n)
    ready = [n for n, k in indeg.items() if k == 0]
    order = []
    while ready:
        n = ready.pop(0)
        order.append(n)
        for c in children[n]:
            indeg[c] -= 1
            if indeg[c] == 0:
                ready.append(c)
    if len(order) != len(graph):
        raise ValueError("依赖成环")
    return order


def fingerprint(node, deps_fp, params_hash):
    payload = f"{node}|{deps_fp}|{params_hash}"
    return hashlib.sha256(payload.encode()).hexdigest()[:6]


print("拓扑序", topo_order(nodes))

done_fp = set()


def run_dag(graph, params_hash):
    fp_now, computed, reused = {}, [], []
    for node in topo_order(graph):
        deps_fp = "|".join(fp_now[d] for d in graph[node])
        fp = fingerprint(node, deps_fp, params_hash)
        fp_now[node] = fp
        if fp in done_fp:
            reused.append(node)
        else:
            done_fp.add(fp)
            computed.append(node)
    return fp_now, computed, reused


_, comp1, _ = run_dag(nodes, "params-v1")
print("第一次计算", comp1)
_, comp2, reused2 = run_dag(nodes, "params-v1")
print("重跑同参数 计算", comp2, "| 复用", len(reused2), "个节点")
_, comp3, _ = run_dag(nodes, "params-v2")
print("改参数后 计算", len(comp3), "个节点")
fp_v1, _, _ = run_dag(nodes, "params-v1")
print("report 节点指纹", fp_v1["report"])


拓扑序 ['load', 'clean', 'factor', 'universe', 'test', 'report']
第一次计算 ['load', 'clean', 'factor', 'universe', 'test', 'report']
重跑同参数 计算 [] | 复用 6 个节点
改参数后 计算 6 个节点
report 节点指纹 99a35a



**读数**：拓扑序是 `load → clean → factor/universe → test → report`。同一套参数重跑，六个节点全部命中指纹、零计算；参数从 `params-v1` 改成 `params-v2` 后，六个节点全部重算。这就是“增量重算”——**不是按时间判断，而是按内容判断**。

**常见错误**：

1. 把它写成“每次全量重跑”，再用人工记忆判断要不要重算。
2. 指纹只包含节点名，不含上游和参数——上游改了却复用旧结果，这是最隐蔽的错。
3. DAG 里放并行写同一个文件的两个节点——依赖图不报错，结果随机坏。

---

## 4. 队列、幂等与状态机

**场景**：凌晨两点批处理跑到一半，机器重启。第二天早上你看到一半任务写着“运行中”，一半写着“完成”，还有几个被重复跑了两次。没有状态机和幂等键，你无法判断哪些结果能直接信、哪些必须重跑。

### 4.1 状态机：只允许合法的迁移

一个作业通常只有六个状态：`pending`（排队）、`running`（执行中）、`success`（成功）、`failed`（失败）、`dead`（死信，放弃）、`cancelled`（人工取消）。合法迁移是有限的：

| 从 | 到 | 触发 |
|---|---|---|
| pending | running / cancelled | 调度 / 取消 |
| running | success / failed | 执行结束 |
| failed | pending / dead | 重试入队 / 达到上限 |
| success、dead、cancelled | （终态） | — |

把这张表写进代码，`success` 的作业就不可能被改成 `running`。这看似多余，实际能挡住“重复提交导致一个作业被跑两次”和“失败作业被误标为成功”。

### 4.2 幂等键：同一件事只做一次

幂等（idempotency）的意思是：同一个请求执行一次和执行十次，结果一样。做法是给每次任务算一个**幂等键**：`(因子 ID, 快照 hash, 参数 hash, 代码 hash)`。入队时先按键查表，已存在就返回原作业号，不再新建。

这样上游重复触发、调度器重发、人工手滑点了两次，都不会产生两份结果。注意：**幂等键必须包含快照 hash**——否则数据换了、键没换，你会拿到旧结果当新结果。

### 4.3 代码：sqlite3 作业表 + 状态机



In [4]:

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("""
CREATE TABLE jobs (
    job_id      TEXT PRIMARY KEY,
    node        TEXT NOT NULL,
    idem_key    TEXT NOT NULL UNIQUE,
    status      TEXT NOT NULL DEFAULT 'pending',
    attempts    INTEGER NOT NULL DEFAULT 0,
    last_error  TEXT,
    updated_at  TEXT NOT NULL
)
""")
conn.commit()

ALLOWED = {
    "pending": {"running", "cancelled"},
    "running": {"success", "failed"},
    "failed": {"pending", "dead"},
    "success": set(),
    "dead": set(),
    "cancelled": set(),
}


def enqueue(job_id, node, idem_key, now="2024-01-05"):
    try:
        cur.execute(
            "INSERT INTO jobs(job_id, node, idem_key, updated_at) VALUES (?, ?, ?, ?)",
            (job_id, node, idem_key, now),
        )
        conn.commit()
        return job_id
    except sqlite3.IntegrityError:
        row = cur.execute("SELECT job_id FROM jobs WHERE idem_key = ?", (idem_key,)).fetchone()
        return row[0] if row else None


def move(job_id, new_status, now="2024-01-05"):
    old = cur.execute("SELECT status FROM jobs WHERE job_id = ?", (job_id,)).fetchone()[0]
    if new_status not in ALLOWED[old]:
        raise ValueError(f"非法迁移 {old} -> {new_status}")
    cur.execute(
        "UPDATE jobs SET status = ?, updated_at = ? WHERE job_id = ? AND status = ?",
        (new_status, now, job_id, old),
    )
    conn.commit()
    return old


print("入队", enqueue("job-001", "load", "snap-a"),
      enqueue("job-002", "compute", "snap-a|rev20|v1"))
print("重复提交同一幂等键 ->", enqueue("job-003", "compute", "snap-a|rev20|v1"))
print("迁移", move("job-001", "running"), "-> running")
print("迁移", move("job-001", "success"), "-> success")
try:
    move("job-001", "running")
except ValueError as exc:
    print("被状态机拒绝:", exc)
print("作业表", cur.execute("SELECT job_id, status, attempts FROM jobs").fetchall())


入队 job-001 job-002
重复提交同一幂等键 -> job-002
迁移 pending -> running
迁移 running -> success
被状态机拒绝: 非法迁移 success -> running
作业表 [('job-001', 'success', 0), ('job-002', 'pending', 0)]



**读数**：第三次提交用的是同一个幂等键，返回的是已有的 `job-002`，没有新增作业；`job-001` 从 `success` 想回到 `running` 时抛出了“非法迁移”，被挡在入口。注意 `move` 的 SQL 里还带了 `AND status = ?`（旧状态做条件），这样两个进程同时改一行时，只有一个能成功——这是数据库层面防止状态被覆盖的常见写法。

**常见错误**：

1. 幂等键只包含任务名——换个数据还是同一个键，拿到旧结果。
2. 用“先查再插”而不加唯一约束——两个进程同时查，都查不到，都插进去。
3. 状态只存一个布尔“是否完成”——分不清“失败可重试”和“死信放弃”。

---



## 5. 失败重试与死信

**场景**：一个节点偶尔因为“数据库连接数满”失败，重跑一次就好；另一个节点因为“字段不存在”失败，重跑一百次也一样。如果对两者都无限重试，产线会被卡死；如果都不重试，偶发故障又会让整个批次白跑。

重试要回答三个问题：**重试几次、隔多久、什么错误才值得重试**。

- 可重试：超时、连接失败、临时限流、下游短暂的 5xx；这类错误重跑大概率成功。
- 不可重试：字段缺失、主键冲突、参数非法、断言失败；这类错误重跑只会重复失败，直接进死信并告警。
- 次数上限：一般 3 次左右。次数越多，失败任务占用资源越久，真正的系统性问题也暴露得越晚。
- 退避：用指数退避（1s、2s、4s、8s……），避免失败瞬间把下游再打一遍。
- 死信（dead letter）：超过上限仍失败，标记为 `dead`，保留最后一次错误，转人工队列，不阻塞其他任务。



In [5]:

class FlakyTask:
    def __init__(self, fail_times):
        self.n = 0
        self.fail_times = fail_times

    def __call__(self):
        self.n += 1
        if self.n <= self.fail_times:
            raise RuntimeError(f"第 {self.n} 次调用失败")
        return f"第 {self.n} 次调用成功"


def run_with_retry(task, max_retry=3):
    trace = []
    for attempt in range(1, max_retry + 2):
        try:
            result = task()
            trace.append((attempt, "success", result))
            return result, trace
        except RuntimeError as exc:
            trace.append((attempt, "failed", str(exc)))
            if attempt > max_retry:
                break
    return "dead_letter", trace


for fail_times in (2, 9):
    task = FlakyTask(fail_times)
    result, trace = run_with_retry(task)
    print(f"预设失败 {fail_times} 次 -> 结果 {result}")
    for attempt, status, msg in trace:
        print("   尝试", attempt, status, msg)
print("指数退避表（秒）", [2 ** (i - 1) for i in range(1, 5)])


预设失败 2 次 -> 结果 第 3 次调用成功
   尝试 1 failed 第 1 次调用失败
   尝试 2 failed 第 2 次调用失败
   尝试 3 success 第 3 次调用成功
预设失败 9 次 -> 结果 dead_letter
   尝试 1 failed 第 1 次调用失败
   尝试 2 failed 第 2 次调用失败
   尝试 3 failed 第 3 次调用失败
   尝试 4 failed 第 4 次调用失败
指数退避表（秒） [1, 2, 4, 8]



**读数**：第一段“预设失败 2 次”重试到第 3 次成功；第二段“预设失败 9 次”在 4 次尝试后确定进入 `dead_letter`（3 次重试 + 1 次初始尝试）。退避表是 1/2/4/8 秒。真实系统里，退避常配合抖动（jitter），避免所有失败任务在同一秒同时重发——**重试不是“多试几次总会好”，而是“在有限预算内给偶发故障一次机会”**。

**常见错误**：

1. 所有异常都重试——不可重试的错误会耗尽重试预算。
2. 不设上限——一个坏任务可能让整个批次永远跑不完。
3. 重试时沿用旧幂等键又直接覆盖状态——重试次数和失败原因都丢了。

---

## 6. 版本与报告 manifest

**场景**：三个月后，业务方拿着报告问：“这个 0.62 的 ICIR 是怎么来的？”你需要在一分钟内回答：哪份数据、哪版代码、哪套参数、哪些作业、谁跑的。

报告 manifest 就是这张“出生证明”。它至少包含六类字段：

| 字段 | 例子 | 为什么必须 |
|---|---|---|
| code_version | `git:7f3a2c1` | 代码可回滚 |
| data_snapshot | `be88967385d09b1a` | 数据可定位 |
| params / params_hash | `window=20` 等 | 参数可复现 |
| jobs | `job-001: success` | 执行可追溯 |
| metrics | `ic=0.041`、`icir=0.62` | 结论可核对 |
| manifest_hash | `2e835f1448498bc9` | 报告本身防篡改 |

manifest 不是“日志”，而是**可校验的声明**。复现函数要做的事是：拿当前数据和参数重新算 hash，和 manifest 里的对比；一致才允许说“这份报告可复现”。不一致时不要覆盖旧报告，保留两份并标出差异。

### 6.1 代码：生成 manifest + 复现校验



In [6]:

params = {"window": 20, "universe": "csi300", "winsor": 3}
params_hash = sha256_text(json.dumps(params, sort_keys=True, ensure_ascii=False))
report = {
    "chapter": 38,
    "factor": "rev20",
    "code_version": "git:7f3a2c1",
    "data_snapshot": snap_hash,
    "params": params,
    "params_hash": params_hash,
    "jobs": {"job-001": "success", "job-002": "success"},
    "metrics": {"ic": 0.041, "icir": 0.62, "turnover": 0.28},
    "generated_at": "2024-01-05",
}
manifest = json.dumps(report, ensure_ascii=False, indent=1, sort_keys=True)
print(manifest[:320].replace("\n", " "), "...")
print("manifest sha256", sha256_text(manifest))


def verify(report, raw_rows, params):
    snap_ok = sha256_text("\n".join(sorted(raw_rows))) == report["data_snapshot"]
    param_ok = (
        sha256_text(json.dumps(params, sort_keys=True, ensure_ascii=False))
        == report["params_hash"]
    )
    return {"快照一致": snap_ok, "参数一致": param_ok, "可复现": snap_ok and param_ok}


print("原样复现", verify(report, raw_rows, params))
print("改一个参数", verify(report, raw_rows, {**params, "window": 60}))
print("数据少一行", verify(report, raw_rows[:-1], params))


{  "chapter": 38,  "code_version": "git:7f3a2c1",  "data_snapshot": "be88967385d09b1a",  "factor": "rev20",  "generated_at": "2024-01-05",  "jobs": {   "job-001": "success",   "job-002": "success"  },  "metrics": {   "ic": 0.041,   "icir": 0.62,   "turnover": 0.28  },  "params": {   "universe": "csi300",   "window": 20 ...
manifest sha256 2e835f1448498bc9
原样复现 {'快照一致': True, '参数一致': True, '可复现': True}
改一个参数 {'快照一致': True, '参数一致': False, '可复现': False}
数据少一行 {'快照一致': False, '参数一致': True, '可复现': False}



**读数**：manifest 的 sha256 是 `2e835f1448498bc9`，其中 `data_snapshot` 就是第 1 节算出的 `be88967385d09b1a`。原样复现三个判断全为真；把 `window` 从 20 改成 60，参数校验失败；删掉一行数据，快照校验失败。**注意两次失败都不会抛异常，而是返回结构化结果**——这样可以批量审计一批报告，而不是让第一份坏报告中断整个检查。

**常见错误**：

1. manifest 只记录指标，不记录数据和参数——报告依然“不可复现”。
2. 参数用字符串拼接算 hash 却不排序——键顺序一变，hash 就变。
3. 把 manifest 写进会被覆盖的临时目录——审计时找不到原件。

---

## 本章小结

### 一张表

| 环节 | 防什么 | 手段 |
|---|---|---|
| 数据契约 | 口径漂移、脏数据 | 字段/主键/时间语义/取值范围断言 |
| PIT 快照 | 偷看未来、快照被覆盖 | `knowledge_date` + 内容 hash 命名 |
| 因子卡 | 没人说得清定义 | 一页纸交接单，随代码升版本 |
| 计算 DAG | 依赖混乱、重复计算 | 拓扑序 + 节点指纹 |
| 队列与状态机 | 重复执行、状态错乱 | 幂等键 + 合法迁移表 |
| 重试与死信 | 偶发故障、坏任务卡死 | 分类重试 + 指数退避 + 上限 |
| 报告 manifest | 数字找不到来源 | 版本/快照/参数/作业/指标 + 自校验 hash |

### 八句话

1. 先定契约，再算因子。
2. 文件名不是版本，内容 hash 才是。
3. PIT 是“当时可见”，不是“最新修订”。
4. 因子卡是交接单，不是论文。
5. DAG 先定依赖，再谈并行。
6. 幂等键要同时含快照和参数。
7. 状态机只允许合法迁移。
8. manifest 让报告可复现、可审计。

---

## 本章习题

**1.** 数据契约至少要写清哪五项内容？

**2.** 为什么快照 hash 之前要先对行排序？

**3.** 因子卡和代码应该是什么关系？

**4.** 节点指纹由哪几部分组成？上游参数变了会怎样？

**5.** 幂等键为什么必须包含快照 hash？

**6.** 哪些错误不应该重试？重试上限一般设几次？

**7.** 为什么 `success` 要设计成终态？

**8.** 报告 manifest 至少要记录哪六类信息？

<details><summary>参考答案</summary>

**1.** 字段名与类型、主键、时间语义（交易日/可见日）、单位与复权口径、缺失规则与取值范围（可再补更新频率）。

**2.** 同一份数据换个导出顺序内容没变，不排序就会算出不同的 hash，被误判成“新快照”。

**3.** 同仓管理：改代码的提交必须同时更新因子卡，方向和口径变更要升版本，卡片和实现不能分家。

**4.** 由节点名、上游节点指纹、参数 hash（再加上代码版本）组成；上游一变，下游指纹全部跟着变，从而触发重算。

**5.** 不包含快照 hash 时，数据换了但幂等键没变，系统会返回旧作业的结果，把旧结果当成新结果。

**6.** 字段缺失、主键冲突、参数非法、断言失败这类确定性错误不重试；上限一般 3 次左右，超过进死信。

**7.** 防止已完成作业被重复改回运行中、被重跑或误标状态；终态代表“这件事已经有结论”。

**8.** 代码版本、数据快照 hash、参数与 params_hash、作业状态、指标，以及 manifest 自身的 hash。

</details>

---

## 参考资料

- Kleppmann, M. *Designing Data-Intensive Applications*，第11–12章（批处理、幂等与可复现）。
- Reis, J. & Housley, M. *Fundamentals of Data Engineering*（数据契约与管道设计）。
- Great Expectations 文档：<https://github.com/great-expectations/great_expectations>
- MLflow 文档：<https://github.com/mlflow/mlflow>
- 第12、19、20章；第40章（行级 PIT 与版本 diff）。
